# 02 — Ensuciado controlado del dataset ScoutAI

**Objetivo:** crear una copia de prueba e introducir problemas de calidad de datos para practicar su detección y limpieza, tal como pide el proyecto.

**No se añaden nulos artificiales a `height_in_cm`**, porque el archivo original ya contiene miles. Se ensucian otras variables para que los errores añadidos sean visibles y medibles.

Los porcentajes son pequeños, pero proporcionan suficientes casos de prueba. Los CSV originales se conservan sin cambios y el proceso se puede repetir con la misma semilla.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np
import math

OFFBALL_DIR = Path('/content/drive/MyDrive/PROYECTO-IGOR-CARMELO-BYRON/OFFBALL')
RAW_DIR = OFFBALL_DIR / 'csv_originales'
DIR_SUCIO = OFFBALL_DIR / 'dataset_prueba_sucio'
DIR_SUCIO.mkdir(parents=True, exist_ok=True)

SEED = 42
rng = np.random.default_rng(SEED)

archivos = ['appearances.csv', 'players.csv', 'player_valuations.csv', 'games.csv', 'clubs.csv']
print('Carpeta de originales existe:', RAW_DIR.exists())
print('Archivos encontrados:', [f.name for f in RAW_DIR.glob('*.csv')])

In [ ]:
# Leer los CSV como texto para conservar los valores al introducir errores.
dfs = {}
for nombre in archivos:
    ruta = RAW_DIR / nombre
    if not ruta.exists():
        raise FileNotFoundError(f'No se encuentra: {ruta}')
    dfs[nombre] = pd.read_csv(ruta, dtype='string', low_memory=False)
    print(f'{nombre}: {len(dfs[nombre]):,} filas | {len(dfs[nombre].columns)} columnas')

## Problemas que introduciremos

| Problema | Cantidad aproximada | Por qué |
|---|---:|---|
| Duplicados en apariciones | 0,5 % | Simular filas repetidas durante una importación. |
| Minutos negativos | 0,5 % | Simular valores imposibles. |
| Valores de mercado ausentes | 1 % | Simular valoraciones que no llegan informadas. |
| Valores de mercado negativos | 0,5 % | Simular valores inválidos. |
| Altura irreal (999 cm) | 1 % | Simular un valor fuera de rango. |
| Fechas inválidas de partidos | 1 % | Simular fechas mal introducidas. |

No se añadirán alturas vacías: ese problema ya existe en los datos originales y lo trataremos durante la limpieza.

In [ ]:
# Solo se crearán copias de las tablas que modificamos. dfs conserva los originales como referencia.
sucios = {}
registro = []

def cantidad_porcentaje(total, porcentaje):
    return min(total, max(1, int(math.ceil(total * porcentaje)))) if total else 0

def elegir_indices(candidatos, n):
    candidatos = pd.Index(candidatos)
    n_real = min(n, len(candidatos))
    if n_real == 0:
        return []
    return rng.choice(candidatos.to_numpy(), size=n_real, replace=False).tolist()

def cambiar_valores(df, tabla, columna, indices, nuevo, problema):
    for idx in indices:
        original = df.at[idx, columna]
        df.at[idx, columna] = nuevo
        registro.append({
            'tabla': tabla,
            'problema': problema,
            'fila_indice': idx,
            'columna': columna,
            'valor_original': original,
            'valor_nuevo': nuevo
        })

In [ ]:
# 1) APARICIONES: duplicados (0,5 %) y minutos negativos (0,5 %).
# Elegimos filas para duplicar y evitamos modificarlas con los minutos negativos,
# para que las filas duplicadas sigan siendo duplicados exactos.

appearances_original = dfs['appearances.csv']  # referencia al original; no se modifica
n_dup = cantidad_porcentaje(len(appearances_original), 0.005)
idx_dup = elegir_indices(appearances_original.index, n_dup)

if 'minutes_played' in appearances_original.columns:
    minutos_num = pd.to_numeric(appearances_original['minutes_played'], errors='coerce')
    candidatos_minutos = minutos_num.index[minutos_num.notna() & (minutos_num >= 0)]
    candidatos_minutos = candidatos_minutos.difference(idx_dup)
    n_minutos = cantidad_porcentaje(len(appearances_original), 0.005)
    idx_minutos = elegir_indices(candidatos_minutos, n_minutos)
    appearances_work = appearances_original.copy()
    cambiar_valores(appearances_work, 'appearances.csv', 'minutes_played',
                    idx_minutos, '-1', 'minutos_negativos')
else:
    appearances_work = appearances_original.copy()
    idx_minutos = []

# Añadir duplicados de filas que no han sido alteradas en minutos.
duplicadas = appearances_work.loc[idx_dup].copy()
if 'appearance_id' in appearances_work.columns:
    for idx in idx_dup:
        valor_id = appearances_work.at[idx, 'appearance_id']
        registro.append({
            'tabla': 'appearances.csv',
            'problema': 'duplicado',
            'fila_indice': idx,
            'columna': 'appearance_id',
            'valor_original': valor_id,
            'valor_nuevo': f'{valor_id} (duplicado añadido)'
        })

sucios['appearances.csv'] = pd.concat([appearances_work, duplicadas], ignore_index=True)
print(f'Filas duplicadas añadidas: {len(duplicadas):,}')
print(f'Minutos negativos introducidos: {len(idx_minutos):,}')

In [ ]:
# 2) PLAYERS: alturas fuera de rango (1 %).
# No añadimos alturas vacías porque ya hay muchas en el archivo original.

players = dfs['players.csv'].copy()
if 'height_in_cm' in players.columns:
    alturas_num = pd.to_numeric(players['height_in_cm'], errors='coerce')
    # Elegimos alturas existentes y plausibles para que el 999 sea un error nuevo.
    candidatos_altura = alturas_num.index[alturas_num.between(130, 220)]
    n_altura = cantidad_porcentaje(len(players), 0.01)
    idx_altura = elegir_indices(candidatos_altura, n_altura)
    cambiar_valores(players, 'players.csv', 'height_in_cm',
                    idx_altura, '999', 'altura_fuera_de_rango')
else:
    idx_altura = []

sucios['players.csv'] = players
print(f'Alturas 999 cm introducidas: {len(idx_altura):,}')

In [ ]:
# 3) VALORACIONES: valores ausentes (1 %) y negativos (0,5 %).
# Los dos grupos no se solapan; los negativos se asignan solo a valores originales válidos.

valuations_original = dfs['player_valuations.csv'].copy()
valuations = valuations_original.copy()

if 'market_value_in_eur' in valuations.columns:
    valor_num = pd.to_numeric(valuations_original['market_value_in_eur'], errors='coerce')
    candidatos_valor = valor_num.index[valor_num.notna() & (valor_num >= 0)]

    n_missing = cantidad_porcentaje(len(valuations_original), 0.01)
    idx_missing = elegir_indices(candidatos_valor, n_missing)
    cambiar_valores(valuations, 'player_valuations.csv', 'market_value_in_eur',
                    idx_missing, pd.NA, 'valor_mercado_ausente')

    candidatos_negativos = candidatos_valor.difference(idx_missing)
    n_neg = cantidad_porcentaje(len(valuations_original), 0.005)
    idx_neg = elegir_indices(candidatos_negativos, n_neg)
    cambiar_valores(valuations, 'player_valuations.csv', 'market_value_in_eur',
                    idx_neg, '-1', 'valor_mercado_negativo')
else:
    idx_missing, idx_neg = [], []

sucios['player_valuations.csv'] = valuations
print(f'Valores de mercado vacíos introducidos: {len(idx_missing):,}')
print(f'Valores de mercado negativos introducidos: {len(idx_neg):,}')

In [ ]:
# 4) GAMES: fechas inválidas (1 %), solo sobre fechas originales válidas.

games_original = dfs['games.csv'].copy()
games = games_original.copy()

if 'date' in games.columns:
    fechas_validas = pd.to_datetime(games_original['date'], errors='coerce')
    candidatos_fecha = fechas_validas.index[fechas_validas.notna()]
    n_fechas = cantidad_porcentaje(len(games_original), 0.01)
    idx_fechas = elegir_indices(candidatos_fecha, n_fechas)
    cambiar_valores(games, 'games.csv', 'date',
                    idx_fechas, '2026-19-45', 'fecha_invalida')
else:
    idx_fechas = []

sucios['games.csv'] = games
print(f'Fechas inválidas introducidas: {len(idx_fechas):,}')

# 5) CLUBS: no se modifica. No necesitamos añadir más tipos de errores.

## Guardar copia ensuciada y registro de cambios

Se guardan los cinco CSV en `dataset_prueba_sucio/`, junto con un registro de cada alteración. Los originales siguen intactos.

In [ ]:
# Añadir clubs.csv sin alteraciones y exportar las cinco tablas.
sucios['clubs.csv'] = dfs['clubs.csv']

# Exportar las copias ensuciadas y el registro exacto de cambios.
for nombre, df in sucios.items():
    df.to_csv(DIR_SUCIO / nombre, index=False)

registro_df = pd.DataFrame(registro)
registro_df.to_csv(DIR_SUCIO / 'registro_ensuciado.csv', index=False)

print('Dataset sucio guardado en:', DIR_SUCIO)
print('Registros de cambios:', len(registro_df))
display(registro_df.groupby('problema').size().reset_index(name='cantidad'))

## Verificación

Comparamos el estado original y la copia ensuciada. La columna **Introducidos** sale del registro de cambios y separa los errores que añadimos de los que ya existían.

In [ ]:
verificaciones = []

def contar_duplicados(df, columna):
    return int(df.duplicated(subset=[columna]).sum()) if columna in df.columns else 0

def contar_negativos(df, columna):
    return int((pd.to_numeric(df[columna], errors='coerce') < 0).sum()) if columna in df.columns else 0

def contar_nulos(df, columna):
    return int(df[columna].isna().sum()) if columna in df.columns else 0

def contar_altura_999(df):
    return int((df['height_in_cm'] == '999').sum()) if 'height_in_cm' in df.columns else 0

def contar_fechas_invalidas(df, columna):
    if columna not in df.columns:
        return 0
    fechas = df[columna]
    return int((fechas.notna() & pd.to_datetime(fechas, errors='coerce').isna()).sum())

def introducidos(problema):
    return int((registro_df['problema'] == problema).sum()) if not registro_df.empty else 0

# Duplicados y minutos
app_original = dfs['appearances.csv']
app_despues = sucios['appearances.csv']
verificaciones.append({
    'comprobacion': 'Duplicados de appearance_id',
    'original': contar_duplicados(app_original, 'appearance_id'),
    'despues': contar_duplicados(app_despues, 'appearance_id'),
    'introducidos': introducidos('duplicado')
})
verificaciones.append({
    'comprobacion': 'Minutos negativos',
    'original': contar_negativos(app_original, 'minutes_played'),
    'despues': contar_negativos(app_despues, 'minutes_played'),
    'introducidos': introducidos('minutos_negativos')
})

# Alturas: mostramos los nulos originales, pero no añadimos nuevos.
players_original = dfs['players.csv']
players_despues = sucios['players.csv']
verificaciones.append({
    'comprobacion': 'Alturas ausentes (no ensuciadas)',
    'original': contar_nulos(players_original, 'height_in_cm'),
    'despues': contar_nulos(players_despues, 'height_in_cm'),
    'introducidos': introducidos('altura_ausente')
})
verificaciones.append({
    'comprobacion': 'Alturas 999 cm',
    'original': contar_altura_999(players_original),
    'despues': contar_altura_999(players_despues),
    'introducidos': introducidos('altura_fuera_de_rango')
})

# Valor de mercado
val_original = dfs['player_valuations.csv']
val_despues = sucios['player_valuations.csv']
verificaciones.append({
    'comprobacion': 'Valores de mercado ausentes',
    'original': contar_nulos(val_original, 'market_value_in_eur'),
    'despues': contar_nulos(val_despues, 'market_value_in_eur'),
    'introducidos': introducidos('valor_mercado_ausente')
})
verificaciones.append({
    'comprobacion': 'Valores de mercado negativos',
    'original': contar_negativos(val_original, 'market_value_in_eur'),
    'despues': contar_negativos(val_despues, 'market_value_in_eur'),
    'introducidos': introducidos('valor_mercado_negativo')
})

# Fechas de partidos
games_original = dfs['games.csv']
games_despues = sucios['games.csv']
verificaciones.append({
    'comprobacion': 'Fechas inválidas en games.csv',
    'original': contar_fechas_invalidas(games_original, 'date'),
    'despues': contar_fechas_invalidas(games_despues, 'date'),
    'introducidos': introducidos('fecha_invalida')
})

verificacion_df = pd.DataFrame(verificaciones)
verificacion_df.to_csv(DIR_SUCIO / 'verificacion_ensuciado.csv', index=False)
display(verificacion_df)
print('Verificación guardada en:', DIR_SUCIO / 'verificacion_ensuciado.csv')

In [ ]:
## Resumen final de esta ejecución

resumen = registro_df.groupby('problema').size().reset_index(name='cantidad')
display(resumen)

n_filas_duplicadas = len(duplicadas)
n_cambios_valor = int((registro_df['problema'] != 'duplicado').sum())
print(f'Filas duplicadas añadidas: {n_filas_duplicadas:,}')
print(f'Valores modificados: {n_cambios_valor:,}')
print(f'Registros totales del registro de cambios: {len(registro_df):,}')
print('Originales intactos. La siguiente fase será limpiar esta copia y comprobar los resultados.')

## Resultado

Se creó una copia de prueba con problemas controlados, el registro de los cambios y la verificación. No se añadieron nulos artificiales de altura porque ya existen en los datos originales. La limpieza se hará en el siguiente notebook.

**Nota:** Google Drive es el espacio temporal de trabajo para Colab. HDFS sigue siendo el almacenamiento central del proyecto; esta copia todavía no significa que se haya cargado en HDFS.